# NB1 — Data EDA + Provenance Log

Structural econometrics pipeline, Phase 1. Authors the provenance log, panel fingerprint, and exploratory diagnostics consumed by NB2 and NB3.

**Status:** skeleton (Task 1c). Cells are authored progressively in later Phase 1 tasks.

> **Gate Verdict:** populated after NB2 and NB3
>
> This admonition is a placeholder. NB3 writes the final gate verdict JSON, and Task 30 auto-renders the human-readable summary into this cell.

## 1. Setup and Data Availability Statement

### Manifest

**Reference.** Ankel-Peters, Brodeur, Connolly and Schwandt (2024), "Data and code availability standards in economics journals," *Q Open* (Social Science Data Editors README template).

**Why used.** The manifest is the first component of the Data Availability Statement (DAS) block: it enumerates every raw source feeding the DuckDB warehouse, together with its row count, observed date range, SHA-256 of the downloaded bytes, and the URL or filesystem path from which it was retrieved.

**Relevance to our results.** Every downstream estimate produced by this notebook chain traces back to a named source in this table. Readers and reviewers can audit whether a specific β̂ or test statistic depends on a source whose fetch is reproducible, whose hash is recorded, and whose row count matches what the pipeline ingested.

**Connection to simulator.** Layer 2 consumers — in particular the RAN payoff bootstrap — re-materialise the DuckDB from the same raw sources when regenerating fitted parameters. The manifest is the contract they read: a source that cannot be re-downloaded from the URL-or-path recorded here cannot be used to refit the simulator's parameters.


In [ ]:
# Bootstrap: make env.py and scripts/econ_query_api.py importable as
# normal modules from this notebook's §1 onward. Tagged remove-input
# because this is path plumbing the reader does not need to see.
#
# Strategy: locate the Colombia/ directory robustly, add it and
# contracts/scripts/ to sys.path, then use plain `import env` / `import
# econ_query_api`. We prefer Path.cwd() because Jupyter sets cwd to the
# notebook's directory by default; we fall back to a walk up the cwd
# tree looking for a directory that contains env.py (covers the case
# where a test runner invokes the notebook from an unexpected cwd).
import sys
from pathlib import Path


def _locate_colombia_dir() -> Path:
    """Find the Colombia/ directory that holds env.py."""
    cwd = Path.cwd().resolve()
    # Fast path: notebook opened from Colombia/ (Jupyter default).
    if (cwd / "env.py").is_file():
        return cwd
    # Walk up: useful when invoked from repo root or contracts/.
    for parent in (cwd, *cwd.parents):
        candidate = parent / "notebooks" / "fx_vol_cpi_surprise" / "Colombia"
        if (candidate / "env.py").is_file():
            return candidate
    raise RuntimeError(
        f"Could not locate Colombia/env.py starting from cwd={cwd}"
    )


_COLOMBIA_DIR = _locate_colombia_dir()
_CONTRACTS_DIR = _COLOMBIA_DIR.parents[2]  # notebooks/../.. = contracts/
_SCRIPTS_DIR = _CONTRACTS_DIR / "scripts"

for _p in (_COLOMBIA_DIR, _SCRIPTS_DIR):
    _p_str = str(_p)
    if _p_str not in sys.path:
        sys.path.insert(0, _p_str)

import env  # noqa: E402  — path plumbing must precede these imports
import econ_query_api  # noqa: E402


In [2]:
import importlib.util
import sys
from pathlib import Path

import duckdb
import pandas as pd

# Load env.py by file path (it is not on sys.path).
_ENV_PATH = Path.cwd() / "env.py"
if not _ENV_PATH.exists():
    # Fallback: notebook may be executed from a different cwd
    _ENV_PATH = (
        Path(__file__).resolve().parent / "env.py"
        if "__file__" in globals()
        else _ENV_PATH
    )

_spec = importlib.util.spec_from_file_location("fx_vol_env", _ENV_PATH)
env = importlib.util.module_from_spec(_spec)
# Register in sys.modules BEFORE exec so that `from __future__ import
# annotations` + `@dataclass(slots=True)` name resolution works.
sys.modules["fx_vol_env"] = env
_spec.loader.exec_module(env)

# Import the query API by file path (scripts/ is not on sys.path).
_API_PATH = env._CONTRACTS_DIR / "scripts" / "econ_query_api.py"
_api_spec = importlib.util.spec_from_file_location("econ_query_api", _API_PATH)
econ_query_api = importlib.util.module_from_spec(_api_spec)
# Same sys.modules registration needed for dataclass(slots=True) resolution.
sys.modules["econ_query_api"] = econ_query_api
_api_spec.loader.exec_module(econ_query_api)

# Open a read-only connection to the structural-econ DuckDB. Kept alive for
# subsequent §1 trios.
conn = duckdb.connect(str(env.DUCKDB_PATH), read_only=True)

# Fetch the manifest and render as a DataFrame for display.
manifest_rows = econ_query_api.get_manifest(conn)
manifest_df = pd.DataFrame(
    [
        {
            "source": r.source,
            "row_count": r.row_count,
            "date_min": r.date_min,
            "date_max": r.date_max,
            "sha256": (r.sha256[:12] + "…") if r.sha256 else None,
            "url_or_path": r.url_or_path,
            "status": r.status,
        }
        for r in manifest_rows
    ]
)
manifest_df


,source,row_count,date_min,date_max,sha256,url_or_path,status
0,banrep:eme,NaN,None,None,None,NaN,unavailable
1,banrep:ibr,4461.0,2008-01-02,2026-04-16,None,https://totoro.banrep.gov.co/nsi-jax-ws/rest/d...,success
2,banrep:intervention,1674.0,1999-12-01,2024-10-04,None,data/raw/banrep_fx_intervention.json,success
3,banrep:trm,8251.0,1991-12-02,2026-04-17,None,https://www.datos.gov.co/resource/32sa-8pi3.js...,success
4,dane:calendar,NaN,None,None,None,NaN,verified
5,dane:calendar,582.0,2002-02-07,2026-04-07,None,NaN,success
6,dane:ipc,NaN,None,None,None,NaN,verified
7,dane:ipc,861.0,1954-07-31,2026-03-31,None,https://suameca.banrep.gov.co/estadisticas-eco...,success
8,dane:ipp,NaN,None,None,None,NaN,verified
9,dane:ipp,322.0,1999-06-30,2026-03-31,None,https://suameca.banrep.gov.co/estadisticas-eco...,success


The manifest above enumerates every raw source currently materialised in the DuckDB warehouse. For each source the table reports:

- **`source`** — canonical short name used as a foreign key by downstream tables.
- **`row_count`** — number of records ingested from that source into its primary landing table.
- **`date_min` / `date_max`** — observed date range of the ingested rows (data coverage, not download timestamp).
- **`sha256`** — truncated hash of the downloaded bytes (first 12 characters shown); the full 64-character digest lives in the `download_manifest` table.
- **`url_or_path`** — public URL or filesystem path from which the source was retrieved, so a fresh clone can reproduce the download.
- **`status`** — provenance lifecycle marker (e.g. `ok`, `imputed`, `failed`).

Any row with a null `sha256` or null `url_or_path` indicates a provenance gap — the pipeline recorded the source but the audit trail is incomplete for that fetch. Such rows are flagged here so readers can decide whether a given downstream computation inherits a provenance gap.


### Date coverage

**Reference.** `econ_query_api._DATE_TABLES` is the canonical enumeration of every warehouse table that carries a date column; each entry pairs a table name with the column used as its time index.

**Why used.** The date-coverage table is the second component of the Data Availability Statement. For every table named in `_DATE_TABLES` it reports row count, observed minimum and maximum date, and the count of rows whose date column is NULL.

**Relevance to our results.** Date coverage bounds the feasible sample window. A table whose observed range ends materially earlier than the others — or carries non-trivial NULL counts in its date column — forces a tighter sample window during Decision #1, and the names appearing here are the candidates that decision evaluates.

**Connection to simulator.** Layer 2 consumers re-estimate parameters against a slice of this same warehouse. The date-coverage table lets them verify that the sample window of a fitted parameter they are about to consume intersects the simulation's target horizon; a table whose coverage no longer spans that horizon invalidates any downstream refit relying on it.


In [3]:
import duckdb
import pandas as pd

# Fresh read-only connection for Trio 2. We do not reuse Trio 1's `conn`
# because the kernel state across a PDF-render execution may have closed
# or re-opened connections; opening our own keeps this cell
# self-contained.
_conn_dc = duckdb.connect(str(env.DUCKDB_PATH), read_only=True)

_coverage = econ_query_api.get_date_coverage(_conn_dc)

coverage_df = pd.DataFrame(
    [
        {
            "table": cov.table,
            "row_count": cov.row_count,
            "date_min": cov.date_min,
            "date_max": cov.date_max,
            "null_count": cov.null_count,
        }
        for cov in _coverage.values()
    ]
).sort_values("table").reset_index(drop=True)

_conn_dc.close()
coverage_df


,table,row_count,date_min,date_max,null_count
0,banrep_ibr_daily,4461,2008-01-02,2026-04-16,0
1,banrep_intervention_daily,1674,1999-12-01,2024-10-04,0
2,banrep_trm_daily,8251,1991-12-02,2026-04-17,0
3,bls_release_calendar,922,1949-03-24,2026-04-10,0
4,daily_panel,5527,2003-01-03,2026-04-17,0
5,dane_ipc_monthly,861,1954-07-01,2026-03-01,0
6,dane_ipp_monthly,322,1999-06-01,2026-03-01,0
7,dane_release_calendar,582,2002-02-07,2026-04-07,0
8,fred_daily,20570,2000-01-03,2026-04-15,0
9,fred_monthly,315,2000-01-01,2026-03-01,0


The table above lists every warehouse table that carries a date column, together with its row count, observed date range, and NULL-date count.

- The **`date_min`** and **`date_max`** columns report the first and last date observed in each table, not the intended coverage window of the underlying source.
- The **`null_count`** column reports how many rows have a NULL value in the table's date column. A non-zero entry means the table contains records whose temporal position is not recorded.
- Tables whose **`date_max`** falls materially before the latest `date_max` seen elsewhere — or whose **`date_min`** falls materially after the earliest `date_min` seen elsewhere — bound the intersection-of-coverage window available to downstream estimation. These are the tables Decision #1 will evaluate when the sample window is pinned.
- Tables with a non-zero **`null_count`** flag a row-level provenance concern in addition to any range mismatch.


## Data Availability Statement

All raw data sources backing the estimates in this notebook chain (NB1 → NB2 → NB3)
were retrieved from public institutional endpoints to a local DuckDB snapshot
(`contracts/data/structural_econ.duckdb`). Retrieval timestamps, observed row
counts, observed date ranges, canonical URLs, and SHA-256 hashes of the
downloaded bytes are recorded row-by-row in the `download_manifest` table and
rendered by the manifest trio above (§1, Trio 1). This statement satisfies the
machine-readability requirement of the Social Science Data Editors (SSDE) 2024
Data and Code Availability Standard: every source named below resolves to a
retrievable URL or archived path, a recorded fetch timestamp, and a
content-addressable hash that permits a replicator to verify byte-level
equivalence against the snapshot used to produce downstream estimates.

All sources are free-to-access. The only endpoint requiring credentials is
FRED, which issues a no-cost personal API key; every other endpoint returns
data without authentication. No proprietary data, paywalled dataset, or
non-public derivative enters this pipeline.

- **Banrep TRM — daily COP/USD Tasa Representativa del Mercado.** Source: Banco de la República de Colombia, published via datos.gov.co (Colombian open-data portal). Retrieved 2026-04-16. Access: free public REST endpoint, no API key. URL: `https://www.datos.gov.co/resource/32sa-8pi3.json`. SHA-256 hash recorded in manifest row `banrep:trm`.
- **Banrep IBR — daily Indicador Bancario de Referencia (Colombian overnight reference rate).** Source: Banco de la República de Colombia, SDMX endpoint. Retrieved 2026-04-16. Access: free public REST endpoint, no API key. URL: `https://totoro.banrep.gov.co/nsi-jax-ws/rest/data/ESTAT,DF_IBR_DAILY_HIST,1.0/all/ALL/`. SHA-256 hash recorded in manifest row `banrep:ibr`.
- **Banrep FX intervention — daily FX intervention flows (spot purchases, sales, options exercised).** Source: Banco de la República de Colombia, archived JSON snapshot at `contracts/data/raw/banrep_fx_intervention.json`. Retrieved 2026-04-16. Access: free public download; no API key. Canonical publisher page: `https://www.banrep.gov.co/en/statistics/exchange-intervention`. SHA-256 hash recorded in manifest row `banrep:intervention`.
- **FRED daily series — VIX (VIXCLS), WTI crude (DCOILWTICO), Brent crude (DCOILBRENTEU).** Source: Federal Reserve Bank of St. Louis FRED API. Retrieved 2026-04-16. Access: free with FRED API key (no-cost personal registration at `https://fred.stlouisfed.org/docs/api/api_key.html`). URL pattern: `https://api.stlouisfed.org/fred/series/observations?series_id={SERIES_ID}`. SHA-256 hashes recorded in manifest rows `fred:VIXCLS`, `fred:DCOILWTICO`, `fred:DCOILBRENTEU`.
- **FRED monthly series — US CPI (CPIAUCSL) and any additional monthly macro controls enumerated in `econ_query_api._FRED_MONTHLY_SERIES`.** Source: Federal Reserve Bank of St. Louis FRED API. Retrieved 2026-04-16. Access: free with FRED API key. URL pattern as above. SHA-256 hash recorded in manifest row `fred:CPIAUCSL`.
- **DANE IPC — monthly Colombian Índice de Precios al Consumidor (headline CPI, 2018=100 base).** Source: Departamento Administrativo Nacional de Estadística (DANE), Colombia, mirrored via SUAMECA (Banrep's historical-series portal). Retrieved 2026-04-16. Access: free public page; extraction performed via headless Playwright because no JSON/CSV endpoint is published. URL: `https://suameca.banrep.gov.co/estadisticas-economicas/informacionSerie/100002/indice_de_precios_al_consumidor_ipc`. SHA-256 hash recorded in manifest row `dane:ipc`.
- **DANE IPP — monthly Colombian Índice de Precios al Productor (producer price index).** Source: DANE, mirrored via SUAMECA. Retrieved 2026-04-16. Access: free public page; extraction performed via headless Playwright. URL: `https://suameca.banrep.gov.co/estadisticas-economicas/informacionSerie/100003/indice_de_precios_al_productor_ipp`. SHA-256 hash recorded in manifest row `dane:ipp`.
- **DANE CPI release calendar — release dates and publication timestamps for IPC and IPP.** Source: DANE publication schedule; no historical calendar is archived online, so the series is algorithmically reconstructed as the 5th business day of each month-after-reference-month (all rows flagged `imputed=TRUE` in the landing table). Retrieved 2026-04-16. Access: derived rule, no external endpoint. Canonical DANE publisher page: `https://www.dane.gov.co/index.php/calendario-estadistico`. SHA-256 hash recorded in manifest row `dane:calendar`.
- **BLS CPI release calendar — US Bureau of Labor Statistics release dates for headline CPI.** Source: US Bureau of Labor Statistics, retrieved via FRED release-calendar endpoint. Retrieved 2026-04-16. Access: free with FRED API key. URL pattern: `https://api.stlouisfed.org/fred/releases/dates?release_id=10`. SHA-256 hash recorded in manifest row `fred:bls_calendar`. Used in NB2 as the comparator calendar when aligning DANE CPI releases against US CPI releases for co-movement diagnostics.

### Reproducibility

Any replicator can verify a byte-level match of this pipeline's inputs against
their own fetch by comparing the SHA-256 hashes enumerated above (resolved
through the manifest table in Trio 1) and the observed date ranges enumerated
by the date-coverage table (Trio 2). The cleaning layer — forthcoming in
Task 13b as `contracts/scripts/cleaning.py` — is the single machine-readable
entry point that transforms the raw landing tables into the analysis panel
consumed by NB1 §2 onward. Raw fetches are isolated behind
`contracts/scripts/econ_query_api.py`, whose calls are deterministic modulo
source-side revisions (FRED vintage revisions and DANE retrospective
corrections are the only documented ways this pipeline's inputs can drift
between two runs at distinct wall-clock dates); any such drift is surfaced by
a changed `downloaded_at` timestamp and, correspondingly, a changed SHA-256
hash in the manifest.


## 2. Panel Construction Audit

### Trio 1 — per-column null and zero counts on `weekly_panel` + `daily_panel`

**Reference.** Social Science Data Editors (SSDE, 2024) Replication Package
Template §3 "Data Quality" — a reproducibility package must document
per-variable completeness of every analysis panel before any estimation
step. Ankel-Peters, Brodeur et al. (2024, *Q Open*, "Reproducing and
replicating development research") explicitly lists "per-column null
audit of the analysis panel" as a replication-bundle minimum.

**Why used.** Before we defend a sample window (Trio 3, Decision #1) or
overlay series for coverage overlap (Trio 2), we need a single table
that shows, per column of each panel, how many rows are NULL and how
many rows equal exactly zero. Silent left-join failures in the
query-API panel-construction layer manifest as elevated null counts on
individual RHS series; silent intervention-dummy mis-alignment
manifests as unexpectedly-high zero counts on `intervention_dummy`.

**Relevance to our results.** A null count above the declared null
policy (populated later in Task 13's `null_policy.yaml`) on the LHS
series `rv_cube_root` or on any pre-committed RHS regressor would
shrink the effective Column-6 OLS sample in NB2 §3, inflating the
standard error on β̂_CPI and weakening the T3b gate. A zero-count
anomaly on `intervention_dummy` would indicate the dummy is not
picking up the Fuentes–Pincheira–Julio–Rincón (2014) intervention
episodes we expect it to encode, which would invalidate the T7
intervention-adequacy test in NB3 §6. Catching either failure mode
here — in Trio 1 — is cheaper than catching it after NB2's fit.

**Connection to simulator.** The weekly-panel null inventory is a
pre-estimation gate on the residuals that seed the Layer 2
filtered-historical-simulation (FHS) bootstrap (Barone-Adesi 2008,
Rev 4 spec §7). If the weekly panel has NULL rows on any residual-
contributing series, the FHS resampling step draws from a residual
empirical distribution that silently excludes those dates — breaking
the stationarity assumption the bootstrap relies on. This audit is
part of the provenance trail that accompanies `nb2_params_full.pkl`'s
residual series into Layer 2.


In [ ]:
import duckdb
import pandas as pd

# Open our own read-only connection for §2 Trio 1. We do not reuse
# §1's `conn` because the kernel state across a PDF-render execution
# may have closed or re-opened connections; opening our own keeps this
# cell self-contained and idempotent.
_conn_pc = duckdb.connect(str(env.DUCKDB_PATH), read_only=True)

# econ_query_api.get_panel_completeness returns a long-form DataFrame
# with columns: panel, column, null_count, zero_count — one row per
# (panel, column) pair, covering both weekly_panel and daily_panel in
# a single call.
completeness_df = econ_query_api.get_panel_completeness(_conn_pc)

# Sort deterministically so diff-review of notebook outputs across
# re-runs is stable. Sorting by (panel, column) matches the cleaning-
# module null-policy YAML ordering (Task 13).
completeness_df = completeness_df.sort_values(
    ["panel", "column"]
).reset_index(drop=True)

_conn_pc.close()
completeness_df


The table above enumerates every column in
`weekly_panel` and `daily_panel` with its null and zero counts. Read
it column-by-column: for each pre-committed series (LHS
`rv_cube_root`, RHS `cpi_surprise`, `us_cpi_surprise`,
`banrep_rate_surprise`, `vix`, `oil_return`, `intervention_dummy`),
the null count is the number of weekly (or daily) observations lost
when that series is used in a regression, and the zero count
disambiguates a genuine missing value from an encoded zero (relevant
for `intervention_dummy`, which encodes "no intervention" as 0 not
NULL).

Any non-zero null count on a pre-committed RHS series is flagged for
resolution in Decision #12 (merge-alignment policy, Task 12 §7);
columns where nulls are structural (e.g. surprises before the AR(1)
warmup window closes) will be addressed in Trio 2 when we compare
per-series coverage to determine the empirical sample start.

This table is a provenance record, not yet a gating criterion —
Trio 3's Decision #1 formalizes the sample-window trim that makes
those null counts analytically acceptable for Column-6 OLS in NB2.


### Trio 2 — coverage overlap and empirical sample start

**Reference.** Social Science Data Editors (SSDE, 2024) Replication
Package Template §2 "Data Scope" — the replication package must state
the empirical sample window the estimation actually uses and defend it
against the claimed window in the manuscript. Ankel-Peters, Brodeur et
al. (2024, *Q Open*, "Reproducing and replicating development
research") §4 "Sample documentation" operationalizes this: the
reproducible sample is the intersection of per-series availability
windows, not the union, and the series whose min-date is the latest is
the *binding* series that determines the effective sample start.

**Why used.** The upstream spec Rev 4 claims a primary sample window
starting 2003-01 for the Colombian COP/USD weekly panel, based on the
Andersen–Bollerslev–Diebold–Vega (2003) AR(1) expanding-window CPI
surprise becoming defined once enough DANE IPC observations have
accumulated. That claim is only valid if **every** raw input series
participating in the Column-6 OLS — TRM, IBR, IPC, IPP, and the FRED
global-risk/US-CPI feeds — is available from at least 2003-01. The
Trio 1 §1 date-coverage table (cell 7 above) already showed IBR begins
2008-01-02; this trio quantifies the implied binding constraint.
Trio 1 §2 (cell 11) also revealed the daily panel's ~22% VIX/oil NULL
rate is a structural Banrep-vs-FRED calendar artifact (weekend
publication + US-only holidays), absorbed by the weekly aggregation,
so coverage overlap is diagnosed at the series level *here* rather
than re-derived from the daily panel's NULL pattern.

**Relevance to our results.** If `empirical_start > 2003-01-01`, any
NB2 Column-6 OLS run over the claimed 2003-01+ window would silently
drop observations (or worse, propagate NaNs into HAC-variance
computation); either way the reported β̂_CPI would correspond to a
sample different from the one documented. Locking `empirical_start`
to the true intersection min-date here lets Trio 3's Decision #1
formalize the sample-window trim as the binding-series start, so NB2
and NB3 can assert-on-load that the weekly panel's `week_start.min()`
equals `empirical_start` without ambiguity.

**Connection to simulator.** The filtered-historical-simulation
(Barone-Adesi 2008, Rev 4 spec §7) bootstrap that Layer 2 uses to
generate RAN payoff paths draws standardized residuals {ẑ_t} from the
Column-6 (and GARCH-X) fit over the empirical window. If the
empirical window is mis-stated upstream — i.e. NB2 fits over 2003-01+
while IBR data only exists from 2008-01 — the residual pool feeding
FHS is contaminated by the GARCH-X fit's implicit NaN-handling
behaviour over 2003–2007, and the resampled volatility paths are
drawn from a distribution that does not correspond to any real
realization of the Colombian FX process. Fixing the empirical window
to the raw-series intersection is therefore not a cosmetic audit
step: it is a correctness prerequisite for the FHS residual pool.


In [ ]:
import duckdb
import pandas as pd

# Fresh read-only connection for §2 Trio 2. Opened and closed within the
# cell so this block can be re-executed idempotently by nbconvert.
_conn_ov = duckdb.connect(str(env.DUCKDB_PATH), read_only=True)

# Raw input series that feed Column-6 OLS in NB2 §3. Calendars
# (dane_release_calendar, bls_release_calendar) are alignment tables,
# not estimation inputs; materialized panels (weekly_panel, daily_panel)
# are downstream of the window we are defending here. The intervention
# dummy is dummy-zero outside its coverage and therefore does not
# constrain the sample start — it is addressed separately in Decision #9.
_RAW_INPUT_TABLES = (
    "banrep_trm_daily",
    "banrep_ibr_daily",
    "dane_ipc_monthly",
    "dane_ipp_monthly",
    "fred_daily",
    "fred_monthly",
)

_coverage_all = econ_query_api.get_date_coverage(_conn_ov)
_coverage_raw = {
    t: _coverage_all[t] for t in _RAW_INPUT_TABLES if t in _coverage_all
}

# Binding series for sample start = table with the latest min-date.
# Binding series for sample end   = table with the earliest max-date.
_binding_start_table, _binding_start_cov = max(
    _coverage_raw.items(), key=lambda kv: kv[1].date_min
)
_binding_end_table, _binding_end_cov = min(
    _coverage_raw.items(), key=lambda kv: kv[1].date_max
)

_empirical_start = _binding_start_cov.date_min
_empirical_end = _binding_end_cov.date_max
_spanning_days = (_empirical_end - _empirical_start).days
_spanning_years = round(_spanning_days / 365.25, 2)

overlap_df = pd.DataFrame(
    [
        {
            "field": "empirical_start",
            "value": _empirical_start.isoformat(),
            "binding_series": _binding_start_table,
        },
        {
            "field": "empirical_end",
            "value": _empirical_end.isoformat(),
            "binding_series": _binding_end_table,
        },
        {
            "field": "spanning_years",
            "value": f"{_spanning_years:.2f}",
            "binding_series": "—",
        },
    ]
)

_conn_ov.close()
overlap_df


The overlap table resolves the empirical sample window to
**2008-01-02 through 2026-03-01**, a span of roughly 18.2 years. The
binding series at the lower edge is **`banrep_ibr_daily`**: IBR (the
Colombian overnight reference rate) was not published before
2008-01-02, so the BanRep rate-surprise regressor is simply undefined
for the 2003–2007 portion of the claimed upstream window. At the
upper edge the binding series is one of the monthly feeds
(`dane_ipc_monthly` / `dane_ipp_monthly` / `fred_monthly`, all ending
2026-03-01), which is expected — monthly releases lag daily feeds by
construction and the exact tie-breaker is immaterial for gate
evaluation.

The 2008-01-02 start is material: the spec Rev 4 "2003-01+" claim
refers to the CPI-surprise AR(1) warmup becoming feasible once DANE
IPC accumulates, but the *full* Column-6 regressor set only becomes
simultaneously observable from 2008-01. Trio 3 will formalize this as
Decision #1 (sample-window lock). The ~18 year span is still well
above the econometric-power threshold implicit in the T3b gate
(β̂_CPI − 1.28·SE > 0 requires n large enough that HAC(4) standard
errors stabilize), so the IBR-binding start does not jeopardize the
identification strategy — it only tightens the honest statement of
what window the estimates describe.
